In [ ]:
#load packages
import os
import numpy as np
import pandas as pd

In [ ]:
#merge temp and gdd data to work for stress indices because we can identify the stages: Booting and Flowering only from gdd
#and to calulate stres indices we need daily temp data

# file paths

CLMATE_DIR = "/group/moniergrp/dbaral"
loca_temp_dir = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data", "LOCA_csv")
loca_gdd_dir = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data","LOCA_gdd")
save_dir = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data", "LOCA_temp_gdd")
final_save_dir = os.path.join(CLIMATE_DIR, "run_project", "intermediate_data", "LOCA_growth_stages")

#get list of files in each dir

gdd_files = sorted([f for f in os.listdir(loca_gdd_dir) if f.endswith(".csv")])
temp_files = sorted([f for f in os.listdir(loca_temp_dir) if f.endswith(".csv")])


for gdd_file in gdd_files:
    base_name = gdd_file.split("_rice")[0]   
    temp_file = [f for f in temp_files if f.startswith(base_name)]

    if not temp_file:
        print(f" No match found for {gdd_file}")
        continue

    temp_file = temp_file[0]

    # read both
    df_gdd  = pd.read_csv(os.path.join(loca_gdd_dir, gdd_file))
    df_temp = pd.read_csv(os.path.join(loca_temp_dir, temp_file))

    # merge 
    merged = pd.merge(df_gdd, df_temp, on=["county","year","date", "month", "day"], how="outer")
    merged = merged.drop(columns = ["spatial_ref"])
    # save
    output_name = base_name + "_temp_gdd_merged.csv"
    merged.to_csv(os.path.join(save_dir, output_name), index=False)
    print(f" Merged: {output_name}")



In [ ]:
files = sorted([f for f in os.listdir(save_dir) if f.endswith(".csv")])

for file in files:
    print(f"processing for {file}")
    temp_gdd_df = pd.read_csv(os.path.join(save_dir, file))
                                                            
    results = []
    
    grouped = temp_gdd_df.groupby(['county', 'year'])
    
    #Loop through each group
    for i, ((county, year), group) in enumerate(grouped):
        group = group.copy() # Get the group DataFrame explicitly
        #print(f"Group {i}: {county}, {year}, {len(group)} rows")
        #print(group.dtypes)

        # convert dates to datetime format
        group['pi_date'] = pd.to_datetime(group['pi_date'], errors='coerce')
        group['hd_date'] = pd.to_datetime(group['hd_date'], errors='coerce')
        group['date']    = pd.to_datetime(group['date'], errors='coerce')
        #get first dates for this group
        first_pi_date = group['pi_date'].dropna().iloc[0]
        first_hd_date = group['hd_date'].dropna().iloc[0]
        #print(first_pi_date)
        #print(first_hd_date)
    
        booting_start = first_pi_date + pd.Timedelta(days = 7)
        booting_end = first_hd_date - pd.Timedelta(days = 7)
    
        flowering_start = first_hd_date - pd.Timedelta(days = 7)
        flowering_end = first_hd_date + pd.Timedelta(days = 7)
    
        grain_fill_start = first_hd_date
        grain_fill_end = first_hd_date + pd.Timedelta(days = 30)
    
        # Initialize the 'stage' column with a default value
    
        group['stage'] = 'growth stage' # Default stage
    
        #filtering for booting stage and naming the stage as booting
    
        group.loc[
            (group['date'] >= booting_start) &
            (group['date'] <= booting_end),
            'stage'
        ] = 'booting'
    
        #filtering for flowering stage and naming the stage column as flowering
        flowering_mask = (
            (group['date'] >= flowering_start) &
            (group['date'] <= first_hd_date)
        )
    
        group.loc[flowering_mask, 'stage'] = 'flowering'
    
        #filtering for overlapping period of flowering and grain fill stage and naming it flowering and grainfill
    
        flowering_grainfill_mask = (
            (group["date"] > first_hd_date) &
            (group["date"] <= flowering_end)
        )
    
        group.loc[flowering_grainfill_mask, 'stage'] = 'flowering_grainfill'
    
    
        #filtering for grain_fil period and naming it as grainfill
    
        grain_fill_mask = (
            (group['date'] > flowering_end) &
            (group['date'] <= grain_fill_end)
        )
    
        group.loc[grain_fill_mask, 'stage'] = 'grain_fill'

    
        # Add tmmx_gf and tmmn_gf columns for grain fill period
        group['tmmx_gf'] = np.nan
        group['tmmn_gf'] = np.nan
    
        # Calculate max and min temperature for the entire grain fill period for this county and year
        grain_fill_period_data = group[group['stage'] == 'grain_fill']
        if not grain_fill_period_data.empty:
            max_tmmx_gf = grain_fill_period_data['tmmx'].max()
            min_tmmn_gf = grain_fill_period_data['tmmn'].min()
            # Assign the calculated max and min to all rows within the grain fill period for this group
            group.loc[group['stage'] == 'grain_fill', 'tmmx_gf'] = max_tmmx_gf
            group.loc[group['stage'] == 'grain_fill', 'tmmn_gf'] = min_tmmn_gf
    
        results.append(group)
    
    #combine all groups
    final_results = pd.concat(results)
    output_name = file.replace(".csv", "_final_growth_stages.csv")
    final_results.to_csv(os.path.join(final_save_dir, output_name), index = False)
    print(f"Saved {output_name}")